In [ ]:
import pandas as pd
import numpy as np

def parse_time_float(time_str):
    """
    Parses time strings to float.
    Returns np.nan for ANY invalid string (e.g., 'FT', 'End', 'Report').
    This ensures garbage rows are dropped and don't become Time 0.0.
    """
    # Convert to string and strip whitespace
    s = str(time_str).strip()

    # 1. Handle Empty/NaN inputs immediately
    if not s or s.lower() in ['nan', 'none', '', 'null']:
        return np.nan

    # 2. Handle Stoppage Time (45+2, 90+4)
    if '+' in s:
        try:
            parts = s.split('+')
            base = int(parts[0])
            extra = int(parts[1]) if parts[1].isdigit() else 0

            # Map for sorting: 45+ -> 45.5, 90+ -> 90.5
            if base == 45:
                return 45.5
            elif base == 90:
                return 90.5
            else:
                return base + 0.5
        except (ValueError, IndexError):
            return np.nan

    # 3. Handle Standard Time
    try:
        return float(s)
    except ValueError:
        # This catches 'FT', 'HT', 'End', etc.
        # Returning NaN ensures these rows are DROPPED later.
        return np.nan

def determine_outcome(home, away):
    if home > away: return 'HOME_WIN'
    elif away > home: return 'AWAY_WIN'
    else: return 'DRAW'

# --- MAIN PROCESS ---

# 1. Load Data
df = pd.read_csv('All_Data_Final_Fixed.csv')

# 2. Parse Time
df['Time_Float'] = df['Time'].apply(parse_time_float)

# 3. CRITICAL FIX: Remove rows with invalid times
# This deletes "Summary" or "Header" rows that were defaulting to 0.0 and breaking the start state
df = df.dropna(subset=['Time_Float'])

# 4. Sort Data
# We sort by Game, then Time, then the original Index (to preserve event order within same minute)
df['Original_Index'] = df.index
df = df.sort_values(by=['Game_ID', 'Time_Float', 'Original_Index'])

# 5. Define Output Bins
bins = [
    (0, "0'"), (5, "5'"), (10, "10'"), (15, "15'"), (20, "20'"),
    (25, "25'"), (30, "30'"), (35, "35'"), (40, "40'"), (45, "45'"),
    (45.5, "HT"), # 45+ events
    (50, "50'"), (55, "55'"), (60, "60'"), (65, "65'"), (70, "70'"),
    (75, "75'"), (80, "80'"), (85, "85'"), (90, "90'"),
    (90.5, "FT")  # 90+ events
]

# Updated state columns for new dataset
state_cols = [
    'Home_Score', 'Away_Score',
    'Home_Red_Count', 'Away_Red_Count',
    'Home_Yellow_Count', 'Away_Yellow_Count',
    'Home_First_Red_Time', 'Away_First_Red_Time',
    'Home_Total_Sub_Count', 'Away_Total_Sub_Count',
    'Home_Defensive_Sub_Count', 'Home_Offensive_Sub_Count', 'Home_Neutral_Sub_Count',
    'Away_Defensive_Sub_Count', 'Away_Offensive_Sub_Count', 'Away_Neutral_Sub_Count'
]

output_rows = []

for game_id, game_events in df.groupby('Game_ID'):

    # Static Info (Match_URL removed; odds added as static metadata)
    first = game_events.iloc[0]
    meta = {
        'Season': first['Season'],
        'Matchweek': first['Matchweek'],
        'Game_Type': first['Game_Type'],
        'Odds_Home_Win': first['Odds_Home_Win'],
        'Odds_Draw': first['Odds_Draw'],
        'Odds_Away_Win': first['Odds_Away_Win']
    }

    # Calculate Outcome (Based on FINAL event of the game)
    last_event = game_events.iloc[-1]
    outcome = determine_outcome(last_event['Home_Score'], last_event['Away_Score'])

    # Generate Bins
    for threshold, label in bins:
        # Get all events that happened up to this threshold
        events_subset = game_events[game_events['Time_Float'] <= threshold]

        if events_subset.empty:
            # No events yet? Score is 0-0, counts are 0, red times are NaN.
            current_state = {col: (np.nan if 'Time' in col else 0) for col in state_cols}
        else:
            # Take the state from the last event in the timeline
            last_valid = events_subset.iloc[-1]
            current_state = {col: last_valid[col] for col in state_cols}

        # Build Row
        row = {
            'Game_ID': game_id,
            **meta,
            'Time': label,
            **current_state,
            'Match_Outcome': outcome
        }
        output_rows.append(row)

# 6. Save
result_df = pd.DataFrame(output_rows)
result_df.to_csv('5-min-breakdown-All.csv', index=False)
print(result_df[['Game_ID', 'Time', 'Home_Score', 'Away_Score',
                  'Home_Red_Count', 'Away_Red_Count',
                  'Home_Yellow_Count', 'Away_Yellow_Count',
                  'Home_First_Red_Time', 'Away_First_Red_Time',
                  'Home_Total_Sub_Count', 'Away_Total_Sub_Count',
                  'Home_Defensive_Sub_Count', 'Home_Offensive_Sub_Count', 'Home_Neutral_Sub_Count',
                  'Away_Defensive_Sub_Count', 'Away_Offensive_Sub_Count', 'Away_Neutral_Sub_Count',
                  'Match_Outcome']].head(25))

      Game_ID Time  Home_Score  Away_Score  Home_Red_Count  Away_Red_Count  \
0   120230101   0'           0           0               0               0   
1   120230101   5'           0           0               0               0   
2   120230101  10'           0           0               0               0   
3   120230101  15'           0           0               0               0   
4   120230101  20'           0           1               0               0   
5   120230101  25'           0           1               0               0   
6   120230101  30'           0           2               0               0   
7   120230101  35'           0           2               0               0   
8   120230101  40'           0           2               0               0   
9   120230101  45'           0           2               0               0   
10  120230101   HT           0           2               0               0   
11  120230101  50'           0           2               0      